# Project Agentic Telegram Test

### Start Ollama Serve

In [ ]:
import os
import subprocess
import time
from urllib.request import urlopen

ollama_environment = os.environ.copy()
ollama_environment["OLLAMA_MODELS"] = "/home/enzo/.ollama/models"

ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    env=ollama_environment,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait briefly for the API to become available
for _ in range(20):
    if ollama_server.poll() is not None:
        raise RuntimeError(
            f"Ollama exited early with code {ollama_server.returncode}"
        )

    try:
        urlopen("http://127.0.0.1:11434/api/tags", timeout=1)
        break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError("Ollama did not start within 10 seconds")

print("Ollama started with PID:", ollama_server.pid)

In [ ]:
# !ollama pull qwen3:4b

In [ ]:
!ollama list

### Stop Ollama Serve

In [2]:
ollama_server.terminate()
ollama_server.wait()
print("Ollama stopped")

NameError: name 'ollama_server' is not defined

## Modal AI Setup

In [2]:
# %cd /home/enzo/project_agentic
# %pip install modal

/home/enzo/project_agentic

[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Selects a workspace and creates an API token

In [3]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "modal", "setup"],
    check=True,
)

Was not able to launch web browsere web browserer
Please go to this URL manually and complete the flow:

]8;id=9701341;https://modal.com/token-flow/tf-VGjQkvoLv8S987gOVwsW3t\https://modal.com/token-flow/tf-VGjQkvoLv8S987gOVwsW3t]8;;\

⠸ Waiting for authentication in the web browser
⠼ Waiting for token flow to complete...omplete...
Web authentication finished successfully!
Token is connected to the modal-agentic-landlord workspace.
Verifying token against https://api.modal.com,https://api.modal2.com
Token verified successfully!
⠋ Storing token
Token written to /home/enzo/.modal.toml in profile modal-agentic-landlord.
→ Run modal skills install to install agent skills


CompletedProcess(args=['/home/enzo/virtual_environments/venv_warehouse/bin/python', '-m', 'modal', 'setup'], returncode=0)

## Database Modal Volumes

#### Tables Created as of wk48

**Property, Billing, and Payment Records**
| Table | What it stores |
|---|---|
| `telegram_groups` | Known Telegram chats: chat ID, title, type, and when last seen. |
| `units` | Unit number, resident display name, default rent/parking, due day, occupancy, and assigned Telegram group. |
| `rent_rates` | Rent amounts effective from a particular month onward. |
| `monthly_bills` | Each unit’s monthly charges, due date, water/electricity readings, utility amounts, and total due. |
| `payment_submissions` | Resident payment claims, screenshot references, review status, landlord notes, confirmed received amount, and payment results. |
| `bill_allocations` | Amounts applied to specific bills, distinguishing payments from advance credit. |
| `credit_transactions` | Advance credit created by overpayments and credit subsequently used. |
| `maintenance_requests` | Reported issues, sender/group, urgency, status, and timestamps. |
| `audit_events` | Records of tracked actions, including who performed them and associated details. |

**Bot Operation**
| Table | What it stores |
|---|---|
| `telegram_inbox` | Incoming cloud updates waiting for processing, including retry information. |
| `processed_updates` | Telegram update IDs already handled, preventing duplicate processing. |
| `telegram_outbox` | Outgoing notifications waiting to be sent, with delivery status and retries. |
| `telegram_cloud_runtime` | Whether cloud processing is enabled for a bot. |
| `billing_reminders` | Which bills have already had their automatic reminder queued. |
| `payment_proof_cleanup` | Screenshot files queued for deletion after review, including cleanup failures. |
| `worker_requests` | Modal write-request receipts, allowing retries without applying the same change twice. Defined in [app/modal_storage.py](/home/enzo/project_agentic/app/modal_storage.py). |

### Database Tables

In [ ]:
import modal
import pandas as pd
from IPython.display import display

worker = modal.Function.from_name(
    "landlord-db-worker-test",
    "database_job",
)

response = worker.remote(
    "dashboard_state",
    dataset="production",
)

if not response.get("ok"):
    raise RuntimeError("Database worker did not confirm the read.")

data = response["result"]

tables = {
    "Units": "units",
    "Telegram groups": "groups",
    "Monthly bills": "bills",
    "Payment submissions": "payments",
    "Maintenance tickets": "maintenance",
}

for title, key in tables.items():
    rows = data.get(key, [])
    print(f"\n{title} — {len(rows)} records")

    if rows:
        display(pd.DataFrame(rows))
    else:
        print("No records.")


Units — 23 records


,id,unit_number,resident_name,due_day,rent_parking_cents,occupancy_status,telegram_group_chat_id,telegram_group_name,telegram_last_seen_at
0,1,102,Benz Ayson,9.0,750000,occupied,-5.503078e+09,Group Benz,2026-11-20 09:32:15
1,2,104,Quinn Hannah,4.0,750000,occupied,NaN,None,None
2,3,201,Jelinda Molina,1.0,800000,occupied,NaN,None,None
3,4,202,Drex,28.0,1100000,occupied,-5.490251e+09,Group Drex,2026-11-24 01:12:01
4,5,203,Abigail,29.0,750000,occupied,NaN,None,None
5,6,204,Junette,24.0,750000,occupied,NaN,None,None
6,7,205,Mark Brian,29.0,750000,occupied,NaN,None,None
7,8,206,Sam,27.0,1000000,occupied,NaN,None,None
8,9,207,Vacant,NaN,700000,vacant,NaN,None,None
9,10,301,Henaro,15.0,800000,occupied,NaN,None,None



Telegram groups — 2 records


,chat_id,title,chat_type,last_seen_at
0,-5503077590,Group Benz,group,2026-11-20 09:32:15
1,-5490251073,Group Drex,group,2026-11-24 01:12:01



Monthly bills — 22 records


,id,unit_id,group_chat_id,group_title,unit_number,resident_name,billing_period,due_date,rent_parking_cents,water_previous,...,electricity_difference,electricity_bill_cents,total_due_cents,allocated_cents,outstanding_cents,status,days_overdue,notes,water_entered,electricity_entered
0,1,1,-5.503078e+09,Group Benz,102,Benz Ayson,2026-11,2026-11-09,750000,163.0,...,None,0,770000,0,770000,overdue,15,None,True,True
1,2,2,NaN,Not connected,104,Quinn Hannah,2026-11,2026-11-04,750000,NaN,...,None,0,750000,0,750000,overdue,20,None,True,True
2,3,3,NaN,Not connected,201,Jelinda Molina,2026-11,2026-11-01,800000,NaN,...,None,0,800000,0,800000,overdue,23,None,True,True
3,4,4,-5.490251e+09,Group Drex,202,Drex,2026-11,2026-11-28,1100000,794.0,...,None,0,1120000,800000,320000,partially_paid,0,None,True,True
4,5,5,NaN,Not connected,203,Abigail,2026-11,2026-11-29,750000,NaN,...,None,0,750000,0,750000,unpaid,0,None,True,True
5,6,6,NaN,Not connected,204,Junette,2026-11,2026-11-24,750000,NaN,...,None,0,750000,0,750000,unpaid,0,None,True,True
6,7,7,NaN,Not connected,205,Mark Brian,2026-11,2026-11-29,750000,NaN,...,None,0,750000,0,750000,unpaid,0,None,True,True
7,8,8,NaN,Not connected,206,Sam,2026-11,2026-11-27,1000000,NaN,...,None,0,1000000,0,1000000,unpaid,0,None,True,True
8,9,10,NaN,Not connected,301,Henaro,2026-11,2026-11-15,800000,NaN,...,None,0,800000,0,800000,overdue,9,None,True,True
9,10,11,NaN,Not connected,302,Tricia,2026-11,2026-11-09,950000,NaN,...,None,0,950000,0,950000,overdue,15,None,True,True



Payment submissions — 6 records


,id,group_chat_id,group_title,submitter_user_id,submitter_name,request_message_id,proof_message_id,telegram_file_id,proof_path,caption,...,amount_received_cents,applied_amount_cents,credit_amount_cents,received_date,or_number,ar_number,status,review_note,created_at,reviewed_at
0,6,-5490251073,Group Drex,5006452143,Lorenzo Altonaga,58,59.0,AgACAgUAAxkBAAM7awTkUg_xUUdfFZxNinfr95KJ7TEAAl...,/storage/production/payment_proofs/payment_6_4...,None,...,NaN,NaN,NaN,None,None,None,pending_review,None,2026-11-24 01:10:56,None
1,5,-5503077590,Group Benz,5006452143,Lorenzo Altonaga,27,28.0,AgACAgUAAxkBAAMcav-2xGKmTE1AXjLw-A7LF08C_HwAAg...,None,None,...,NaN,NaN,NaN,None,None,None,rejected,not good this is for oil la,2026-11-20 02:54:57,2026-11-20 02:57:17
2,4,-5490251073,Group Drex,5006452143,Lorenzo Altonaga,84,85.0,AgACAgUAAxkBAANVav3VMpdpDadRhqVfI7lgD96tXKQAAm...,None,None,...,800000.0,800000.0,0.0,2026-11-19,None,None,approved,None,2026-11-18 16:38:05,2026-11-18 16:46:06
3,3,-5503077590,Group Benz,5006452143,Lorenzo Altonaga,69,70.0,AgACAgUAAxkBAANBav2ogHe7pzvvrLP99oXaBP2HSucAAu...,None,None,...,NaN,NaN,NaN,None,None,None,rejected,not correct,2026-11-18 13:37:18,2026-11-18 13:37:38
4,2,-5503077590,Group Benz,5006452143,Lorenzo Altonaga,63,65.0,AgACAgUAAxkBAANBav2ogHe7pzvvrLP99oXaBP2HSucAAu...,None,@EnzoGPTest_bot,...,NaN,NaN,NaN,None,None,None,approved,None,2026-11-18 13:26:43,2026-11-18 13:31:43
5,1,-5503077590,Group Benz,1222115255,Kler,54,NaN,None,None,None,...,NaN,NaN,NaN,None,None,None,awaiting_proof,None,2026-11-18 13:13:25,None



Maintenance tickets — 4 records


,id,resident_id,group_chat_id,group_title,submitter_user_id,submitter_name,telegram_update_id,description,urgency,status,created_at
0,4,NaN,-5.503078e+09,Group Benz,5.006452e+09,Lorenzo Altonaga,539765280,Resident reporting a broken window.,normal,open,2026-11-20 04:36:20
1,3,NaN,-5.490251e+09,Group Drex,5.006452e+09,Lorenzo Altonaga,539765279,Resident reports paint peeling from walls.,normal,open,2026-11-20 04:32:53
2,2,NaN,-5.503078e+09,Group Benz,1.222115e+09,Kler,292226857,Resident reports payment of rent for two month...,normal,resolved,2026-11-18 13:17:05
3,1,1.0,NaN,Authorized resident 5006452143,NaN,Authorized resident 5006452143,292226845,Resident reports a broken shower.,normal,closed,2026-11-18 06:49:32


In [ ]:
tables

{'Units': 'units',
 'Telegram groups': 'groups',
 'Monthly bills': 'bills',
 'Payment submissions': 'payments',
 'Maintenance tickets': 'maintenance'}

Creating Table Command

In [6]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "modal",
        "volume", "create",
        "landlord-data",
        "--version=2",
    ],
    check=True,
)

Created Volume 'landlord-data' in environment ''. 

Code example:

                                                                                
@app.function(volumes={"/my_vol": modal.Volume.from_name("landlord-data")})     
def some_func():                                                                
    os.listdir("/my_vol")                                                       
                                                                                


CompletedProcess(args=['/home/enzo/virtual_environments/venv_warehouse/bin/python', '-m', 'modal', 'volume', 'create', 'landlord-data', '--version=2'], returncode=0)

Viewing Volumes Created

In [7]:
subprocess.run(
    [sys.executable, "-m", "modal", "volume", "list"],
    check=True,
)

┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Name          ┃ Created at           ┃ Created by ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ landlord-data │ 2026-11-19 22:32 PST │ enzoalto   │
└───────────────┴──────────────────────┴────────────┘


CompletedProcess(args=['/home/enzo/virtual_environments/venv_warehouse/bin/python', '-m', 'modal', 'volume', 'list'], returncode=0)

In [9]:
subprocess.run(
    [
        sys.executable, "-m", "modal",
        "volume", "ls", "landlord-data", "/",
    ],
    check=True,
)

CompletedProcess(args=['/home/enzo/virtual_environments/venv_warehouse/bin/python', '-m', 'modal', 'volume', 'ls', 'landlord-data', '/'], returncode=0)

# Main App Run

## Version 3: Modal Worker
Modal Changes only reflect when Redeployed with the code below. Similar to Docker

In [1]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/enzo/project_agentic")
worker_file = PROJECT_ROOT / "modal_worker.py"

if not worker_file.is_file():
    raise FileNotFoundError(f"Worker file not found: {worker_file}")

subprocess.run(
    [
        sys.executable,
        "-m",
        "modal",
        "deploy",
        str(worker_file),
        "--strategy=recreate",
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

print("Worker redeployed. Existing Modal database storage is preserved.")

⠸ Creating objects.....
├── ⠋
⠦ Creating objects...
├── ⠸ Creating mount /home/enzo/project_agentic/modal_worker.py: Finalizing 
│   index of 1 files
⠏ Creating objects...
├── ✓ Uploaded /home/enzo/project_agentic/modal_worker.py
⠹ Creating objects...ge:app: Uploaded 18/20 missing files
├── ✓ Uploaded /home/enzo/project_agentic/modal_worker.py
⠴ Creating objects...ge:app: Uploaded 18/20 missing files
├── ✓ Uploaded /home/enzo/project_agentic/modal_worker.py
⠇ Creating objects...ge:app: Uploaded 18/20 missing files
├── ✓ Uploaded /home/enzo/project_agentic/modal_worker.py
⠙ Creating objects...ge:app: Finalizing index of 20 files
├── ✓ Uploaded /home/enzo/project_agentic/modal_worker.py
├── ✓ Uploaded PythonPackage:app
⠴ Creating objects...
├── ✓ Uploaded /home/enzo/project_agentic/modal_worker.py
├── ✓ Uploaded PythonPackage:app
⠴ Creating objects...
├── ✓ Uploaded /home/enzo/project_agentic/modal_worker.py
├── ✓ Uploaded PythonPackage:app
└── ✓ Created Function database_job.
✓ Created 

Switching to Modal
- Starts Telegram Bot on Modal
- Alternative to local Start up

In [6]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "scripts.configure_modal_telegram",
     "--activate", "--confirm-local-stopped"],
    cwd="/home/enzo/project_agentic",
    check=True,
)

{'enabled': True, 'webhook_url': 'https://modal-agentic-landlord--landlord-telegram-webhook.us-east.modal.run', 'pending_updates': 0, 'ai_mode': 'modal', 'last_delivery_error': False}


CompletedProcess(args=['/home/enzo/virtual_environments/venv_warehouse/bin/python', '-m', 'scripts.configure_modal_telegram', '--activate', '--confirm-local-stopped'], returncode=0)

Switching to Local (Experimentation)

In [ ]:
subprocess.run(
    [sys.executable, "-m", "scripts.configure_modal_telegram",
     "--deactivate"],
    cwd="/home/enzo/project_agentic",
    check=True,
)

## Version 2: Local Dashboard
Online Dashboard: https://landlord-riel.streamlit.app/
- Changes are reflected live based on Github.
- Changes only happen when pushed and synchronized on github



Start Application

In [3]:
import os
import subprocess
import sys
import time
from pathlib import Path
from urllib.request import urlopen

# Find the project directory.
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "dashboard.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "dashboard.py").exists():
    raise FileNotFoundError(
        "Could not find dashboard.py. Open Jupyter from the "
        "project_agentic directory or its notebooks directory."
    )

DASHBOARD_URL = "http://127.0.0.1:8501"
HEALTH_URL = f"{DASHBOARD_URL}/_stcore/health"


def dashboard_is_online():
    try:
        with urlopen(HEALTH_URL, timeout=2) as response:
            return response.status == 200
    except Exception:
        return False


if dashboard_is_online():
    print("Dashboard is already online:")
    print(DASHBOARD_URL)

elif (
    "streamlit_process" in globals()
    and streamlit_process.poll() is None
):
    print("Dashboard process is already running.")
    print("PID:", streamlit_process.pid)
    print("URL:", DASHBOARD_URL)

else:
    # Prefer the project virtual environment.
    project_python = PROJECT_ROOT / ".venv" / "bin" / "python"
    python_executable = (
        str(project_python)
        if project_python.exists()
        else sys.executable
    )

    log_path = PROJECT_ROOT / "data" / "streamlit_dashboard.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)

    streamlit_log = open(
        log_path,
        mode="a",
        encoding="utf-8",
        buffering=1,
    )

    dashboard_environment = os.environ.copy()
    dashboard_environment["PYTHONUNBUFFERED"] = "1"
    dashboard_environment["STREAMLIT_BROWSER_GATHER_USAGE_STATS"] = "false"

    streamlit_process = subprocess.Popen(
        [
            python_executable,
            "-m",
            "streamlit",
            "run",
            str(PROJECT_ROOT / "dashboard.py"),
            "--server.address",
            "127.0.0.1",
            "--server.port",
            "8501",
            "--server.headless",
            "true",
        ],
        cwd=str(PROJECT_ROOT),
        env=dashboard_environment,
        stdout=streamlit_log,
        stderr=subprocess.STDOUT,
        text=True,
    )

    # Wait briefly for Streamlit to become available.
    for _ in range(15):
        if streamlit_process.poll() is not None:
            break
        if dashboard_is_online():
            break
        time.sleep(1)

    if dashboard_is_online():
        print("Dashboard started successfully.")
        print("PID:", streamlit_process.pid)
        print("URL:", DASHBOARD_URL)
        print("Log:", log_path)
    else:
        exit_code = streamlit_process.poll()

        if exit_code is None:
            print("Dashboard is still starting.")
            print("Check:", DASHBOARD_URL)
        else:
            streamlit_log.close()
            print("Dashboard exited with code:", exit_code)
            print("\nRecent log output:")
            print(log_path.read_text(encoding="utf-8")[-4000:])

Dashboard started successfully.
PID: 21625
URL: http://127.0.0.1:8501
Log: /home/enzo/project_agentic/data/streamlit_dashboard.log


Stop Dashboard

In [2]:
import os
import signal
import subprocess
import time
from pathlib import Path

PORT = 8501

# Stop the process tracked by this notebook.
if (
    "streamlit_process" in globals()
    and streamlit_process.poll() is None
):
    streamlit_process.terminate()

    try:
        streamlit_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        streamlit_process.kill()
        streamlit_process.wait()

if "streamlit_log" in globals() and not streamlit_log.closed:
    streamlit_log.close()


def listening_pids():
    result = subprocess.run(
        ["lsof", "-t", f"-iTCP:{PORT}", "-sTCP:LISTEN"],
        capture_output=True,
        text=True,
    )
    return {
        int(pid)
        for pid in result.stdout.split()
        if pid.isdigit()
    }


# Find and stop Streamlit listeners left by other notebook sessions.
try:
    for pid in listening_pids():
        try:
            command = (
                Path(f"/proc/{pid}/cmdline")
                .read_bytes()
                .replace(b"\0", b" ")
                .decode(errors="replace")
            )

            if "streamlit" not in command or "dashboard.py" not in command:
                print(
                    f"Port {PORT} belongs to another application "
                    f"(PID {pid}); left running."
                )
                continue

            os.kill(pid, signal.SIGTERM)

            for _ in range(50):
                if pid not in listening_pids():
                    break
                time.sleep(0.1)
            else:
                os.kill(pid, signal.SIGKILL)

            print(f"Stopped dashboard PID {pid}.")

        except (ProcessLookupError, FileNotFoundError):
            pass

    remaining = listening_pids()

    if remaining:
        print(f"Port {PORT} is still occupied by PID(s): {sorted(remaining)}")
    else:
        print("Dashboard stopped. Rerun the dashboard-start cell.")

except FileNotFoundError:
    print("Port checking requires lsof. Install it in your WSL terminal:")
    print("sudo apt install lsof")

Dashboard stopped. Rerun the dashboard-start cell.


## Version 1: Local Qwen Model
- telegram bot functions only update when stopped and started again

Start on Bot

In [3]:
import os
import re
import subprocess
import sys
import threading
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "run_bot.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "run_bot.py").exists():
    raise FileNotFoundError(
        "Could not find run_bot.py. "
        "Open the notebook from the project_agentic workspace."
    )

log_path = PROJECT_ROOT / "data" / "telegram_bot.log"
log_path.parent.mkdir(parents=True, exist_ok=True)

# Prevent Telegram bot tokens from appearing in notebook/log output.
token_pattern = re.compile(
    r"bot\d+:[A-Za-z0-9_-]+"
)


def stream_bot_output(process, output_path):
    """Copy subprocess output to both Jupyter and a log file."""

    with output_path.open(
        mode="a",
        encoding="utf-8",
        buffering=1,
    ) as log_file:
        for line in process.stdout:
            safe_line = token_pattern.sub(
                "bot<REDACTED>",
                line,
            )

            print(safe_line, end="", flush=True)
            log_file.write(safe_line)

    exit_code = process.wait()
    print(
        f"\nTelegram bot process exited "
        f"with code {exit_code}."
    )


if (
    "telegram_bot_process" in globals()
    and telegram_bot_process.poll() is None
):
    print(
        "Telegram bot is already running with PID:",
        telegram_bot_process.pid,
    )
    print(
        "Stop it first if it was started using "
        "the previous non-streaming cell."
    )
else:
    bot_environment = os.environ.copy()
    bot_environment["PYTHONUNBUFFERED"] = "1"

    telegram_bot_process = subprocess.Popen(
        [
            sys.executable,
            "-u",
            str(PROJECT_ROOT / "run_bot.py"),
        ],
        cwd=str(PROJECT_ROOT),
        env=bot_environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    telegram_output_thread = threading.Thread(
        target=stream_bot_output,
        args=(telegram_bot_process, log_path),
        daemon=True,
    )
    telegram_output_thread.start()

    time.sleep(2)

    if telegram_bot_process.poll() is None:
        print(
            "Telegram bot started with PID:",
            telegram_bot_process.pid,
        )
        print("Output will stream below this cell.")
        print("Log file:", log_path)
    else:
        print(
            "Telegram bot exited during startup. "
            "Review the output above."
        )

Telegram bot started with PID: 8633
Output will stream below this cell.
Log file: /home/enzo/project_agentic/data/telegram_bot.log


In [16]:
from pathlib import Path
import re

log = Path("/home/enzo/project_agentic/data/telegram_bot.log")
text = log.read_text(encoding="utf-8")[-3000:]

# Hide Telegram tokens if an error includes a request URL.
text = re.sub(r"bot\d+:[A-Za-z0-9_-]+", "bot<REDACTED>", text)
print(text)

ACTED>/getUpdates
Landlord Telegram bot is online. Press Ctrl+C to stop.
Landlord Telegram bot is online. Press Ctrl+C to stop.
Telegram connection error: HTTPSConnectionPool(host='api.telegram.org', port=443): Read timed out. (read timeout=40)
Landlord Telegram bot is online. Press Ctrl+C to stop.
Processing Telegram update 292226848 from Lorenzo Altonaga in -5503077590
Classified update 292226848 as general_chat
Processing Telegram update 292226849 from Lorenzo Altonaga in -5503077590
Classified update 292226849 as general_chat
Processing Telegram update 292226850 from Lorenzo Altonaga in 5006452143
Classified update 292226850 as general_chat
Processing Telegram update 292226851 from Lorenzo Altonaga in -5503077590
Classified update 292226851 as general_chat
Processing Telegram update 292226853 from Kler in -5503077590
Classified update 292226853 as payment_submission
Processing Telegram update 292226857 from Kler in -5503077590
Classified update 292226857 as maintenance_request
Proc

Stop Bot

In [5]:
import subprocess

if (
    "telegram_bot_process" not in globals()
    or telegram_bot_process.poll() is not None
):
    print("Telegram bot is not running.")
else:
    print("Stopping Telegram bot...")

    telegram_bot_process.terminate()

    try:
        telegram_bot_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        print("Graceful stop timed out; killing process.")
        telegram_bot_process.kill()
        telegram_bot_process.wait()

    if "telegram_output_thread" in globals():
        telegram_output_thread.join(timeout=3)

    print("Telegram bot stopped.")

Telegram bot is not running.


View Stored Tickets

In [10]:
import sqlite3
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

database_path = PROJECT_ROOT / "data" / "landlord.db"

with sqlite3.connect(database_path) as connection:
    connection.row_factory = sqlite3.Row

    rows = connection.execute("""
        SELECT
            m.id AS ticket_id,
            r.name AS resident,
            r.unit,
            m.description AS issue,
            m.urgency,
            m.status,
            datetime(
                m.created_at,
                '+8 hours'
            ) AS created_at_manila
        FROM maintenance_requests AS m
        JOIN residents AS r
            ON r.id = m.resident_id
        ORDER BY m.id DESC
    """).fetchall()

tickets = [dict(row) for row in rows]
tickets

[{'ticket_id': 1,
  'resident': 'Authorized resident 5006452143',
  'unit': 'Unassigned',
  'issue': 'Resident reports a broken shower.',
  'urgency': 'normal',
  'status': 'open',
  'created_at_manila': '2026-11-18 14:49:32'}]

# Test Runs

In [13]:
import json
from urllib.request import Request, urlopen

payload = {
    # "model": "qwen3.5:4b-q4_K_M",

    "model": "qwen3:4b",
    "messages": [
        {
            "role": "user",
            "content": "Explain an agentic workflow in three short bullets."
        }
    ],
    "stream": False
}

request = Request(
    "http://127.0.0.1:11434/api/chat",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with urlopen(request, timeout=300) as response:
    result = json.load(response)

print(result["message"]["content"])

HTTPError: HTTP Error 404: Not Found

In [ ]:
import json
from urllib.request import Request, urlopen

payload = {
    "model": "qwen3.5:4b-q4_K_M",

    # "model": "qwen3:4b",
    "messages": [
        {
            "role": "user",
            "content": "Explain an agentic workflow in three short bullets."
        }
    ],
    "stream": False
}

request = Request(
    "http://127.0.0.1:11434/api/chat",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with urlopen(request, timeout=300) as response:
    result = json.load(response)

print(result["message"]["content"])

# Telegram

In [ ]:
%run read_telegram.py

## Bot Reading Sandbox

In [22]:
import sqlite3
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "landlord.db").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

database_path = PROJECT_ROOT / "data" / "landlord.db"

with sqlite3.connect(database_path) as connection:
    connection.row_factory = sqlite3.Row

    chats = connection.execute(
        """
        SELECT
            chat_id,
            title,
            chat_type,
            first_seen_at,
            last_seen_at
        FROM telegram_groups
        ORDER BY last_seen_at DESC
        """
    ).fetchall()

if not chats:
    print("No chats recorded yet.")
    print("Mention the updated bot in a group, then run this cell again.")
else:
    for chat in chats:
        print("Chat ID:", chat["chat_id"])
        print("Title:", chat["title"])
        print("Type:", chat["chat_type"])
        print("Last seen:", chat["last_seen_at"])
        print("-" * 40)

No chats recorded yet.
Mention the updated bot in a group, then run this cell again.


In [20]:
if (
    "telegram_bot_process" in globals()
    and telegram_bot_process.poll() is None
):
    telegram_bot_process.terminate()
    telegram_bot_process.wait(timeout=10)

    if "telegram_bot_log" in globals() and not telegram_bot_log.closed:
        telegram_bot_log.close()

    print("Old bot process stopped.")
else:
    print("No bot process found in this notebook kernel.")

No bot process found in this notebook kernel.


Connected to Ollama

In [ ]:
import os
import time
import requests
from collections import defaultdict
from dotenv import load_dotenv

load_dotenv()

# ----------------------------
# Configuration
# ----------------------------

TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
if not TOKEN:
    raise ValueError("Missing TELEGRAM_BOT_TOKEN in .env")

# Optional: add TELEGRAM_ALLOWED_CHAT_ID to .env to restrict access.
ALLOWED_CHAT_ID = os.getenv("TELEGRAM_ALLOWED_CHAT_ID")

TELEGRAM_URL = f"https://api.telegram.org/bot{TOKEN}"
OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "qwen3.5:4b-q4_K_M"

SYSTEM_PROMPT = """
You are a helpful Telegram assistant.
Give clear, concise answers.
Use plain text unless the user requests code or detailed formatting.
""".strip()

# Stores separate conversation histories for each Telegram chat.
conversation_history = defaultdict(list)

session = requests.Session()


# ----------------------------
# Telegram functions
# ----------------------------

def send_telegram_message(chat_id, text):
    """Send text, splitting messages that exceed Telegram's limit."""

    text = text.strip() or "(No response generated.)"

    # Telegram allows up to 4096 characters; use 4000 for safety.
    chunks = [
        text[position:position + 4000]
        for position in range(0, len(text), 4000)
    ]

    for chunk in chunks:
        response = session.post(
            f"{TELEGRAM_URL}/sendMessage",
            json={
                "chat_id": chat_id,
                "text": chunk,
            },
            timeout=30,
        )
        response.raise_for_status()


# ----------------------------
# Ollama function
# ----------------------------

def ask_ollama(chat_id, user_text):
    history = conversation_history[chat_id]

    history.append({
        "role": "user",
        "content": user_text,
    })

    # Keep only the six most recent user/assistant exchanges.
    history[:] = history[-12:]

    response = session.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": MODEL,
            "messages": [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT,
                },
                *history,
            ],
            "stream": False,
            "think": False,
            "keep_alive": "10m",
            "options": {
                "num_ctx": 8192,
                "num_predict": 600,
                "temperature": 0.7,
            },
        },
        timeout=300,
    )
    response.raise_for_status()

    result = response.json()
    assistant_text = result["message"]["content"].strip()

    history.append({
        "role": "assistant",
        "content": assistant_text,
    })

    history[:] = history[-12:]
    return assistant_text


# ----------------------------
# Preflight check
# ----------------------------

health_response = session.get(
    f"{OLLAMA_URL}/api/tags",
    timeout=10,
)
health_response.raise_for_status()

installed_models = [
    model["name"]
    for model in health_response.json().get("models", [])
]

if MODEL not in installed_models:
    raise RuntimeError(
        f"{MODEL} is not available. Installed models: {installed_models}"
    )

print(f"Ollama is ready: {MODEL}")

if ALLOWED_CHAT_ID:
    print(f"Access restricted to Telegram chat: {ALLOWED_CHAT_ID}")
else:
    print("Warning: no TELEGRAM_ALLOWED_CHAT_ID is configured.")


# ----------------------------
# Bot polling loop
# ----------------------------

def start_ollama_bot():
    offset = None

    print("Telegram Ollama bot is online.")
    print("Send /reset to clear conversation memory.")
    print("Interrupt this Jupyter cell to stop the bot.")

    while True:
        try:
            params = {
                "timeout": 25,
                "allowed_updates": '["message"]',
            }

            if offset is not None:
                params["offset"] = offset

            response = session.get(
                f"{TELEGRAM_URL}/getUpdates",
                params=params,
                timeout=35,
            )
            response.raise_for_status()

            result = response.json()

            if not result.get("ok"):
                raise RuntimeError(result)

            for update in result.get("result", []):
                # Confirm this update so Telegram does not send it again.
                offset = update["update_id"] + 1

                message = update.get("message")
                if not message or not message.get("text"):
                    continue

                chat_id = message["chat"]["id"]
                user_text = message["text"].strip()
                first_name = message.get("from", {}).get(
                    "first_name",
                    "User",
                )

                if (
                    ALLOWED_CHAT_ID
                    and str(chat_id) != str(ALLOWED_CHAT_ID)
                ):
                    print(f"Ignored unauthorized chat: {chat_id}")
                    continue

                print(f"{first_name} ({chat_id}): {user_text}")

                if user_text.lower() == "/reset":
                    conversation_history.pop(chat_id, None)
                    send_telegram_message(
                        chat_id,
                        "Conversation memory cleared.",
                    )
                    continue

                if user_text.lower() == "/start":
                    send_telegram_message(
                        chat_id,
                        "Hello! I am connected to a local Qwen model. "
                        "Send me a message, or use /reset to clear memory.",
                    )
                    continue

                try:
                    reply = ask_ollama(chat_id, user_text)
                    send_telegram_message(chat_id, reply)
                    print(f"Qwen: {reply[:200]}")

                except requests.RequestException as exc:
                    print("Ollama request failed:", exc)
                    send_telegram_message(
                        chat_id,
                        "Sorry, the local language model encountered "
                        "an error. Please try again.",
                    )

        except KeyboardInterrupt:
            print("\nTelegram Ollama bot stopped.")
            break

        except requests.RequestException as exc:
            print("Telegram connection error:", exc)
            time.sleep(3)


start_ollama_bot()

Simply Reads the Texts from Telegram

In [14]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")

if not TOKEN:
    raise ValueError("TELEGRAM_BOT_TOKEN not found in .env")

url = f"https://api.telegram.org/bot{TOKEN}/getUpdates"

response = requests.get(url, timeout=15)
response.raise_for_status()

data = response.json()

for update in data.get("result", []):
    message = update.get("message", {})
    print("Sender:", message.get("from", {}).get("first_name"))
    print("Chat ID:", message.get("chat", {}).get("id"))
    print("Text:", message.get("text"))

if not data.get("result"):
    print("No pending updates found.")

No pending updates found.


In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
if not TOKEN:
    raise ValueError("Missing TELEGRAM_BOT_TOKEN in .env")

BASE_URL = f"https://api.telegram.org/bot{TOKEN}"

def send_reply(chat_id, text):
    response = requests.post(
        f"{BASE_URL}/sendMessage",
        json={"chat_id": chat_id, "text": text},
        timeout=15
    )
    response.raise_for_status()
    return response.json()

def start_bot():
    offset = None
    print("Bot is online! Send it a Telegram message.")
    print("Interrupt the Jupyter cell to stop.")

    while True:
        try:
            params = {"timeout": 20}
            if offset is not None:
                params["offset"] = offset

            response = requests.get(
                f"{BASE_URL}/getUpdates",
                params=params,
                timeout=30
            )
            response.raise_for_status()
            result = response.json()

            if not result.get("ok"):
                raise RuntimeError(result)

            for update in result["result"]:
                offset = update["update_id"] + 1
                message = update.get("message")

                if not message or not message.get("text"):
                    continue

                chat_id = message["chat"]["id"]
                user_text = message["text"]

                print(f"Received: {user_text}")

                # Echo the message back to the sender
                reply = f"You said: {user_text}"
                send_reply(chat_id, reply)

                print(f"Replied: {reply}")

        except KeyboardInterrupt:
            print("\nBot stopped.")
            break
        except requests.RequestException as exc:
            print("Telegram connection error:", exc)

start_bot()
